# 11 — Gaussian Processes
**References:** Rasmussen & Williams (2006) *GPML* Ch. 1–5 · Murphy (2022) *PML2* Ch. 17

## Narrative thread
```
GP as prior over functions -> Kernel functions -> GP regression -> Hyperparameter learning -> Classification
```

## 1. Gaussian processes as function priors

A **Gaussian process** is a collection of random variables such that any finite subset has a joint Gaussian distribution.

A GP is fully specified by a **mean function** $m(x) = E[f(x)]$ and a **covariance (kernel) function** $k(x, x') = \text{Cov}(f(x), f(x'))$:
$$f \sim \mathcal{GP}(m, k)$$

For any finite set of inputs $\mathbf{x} = (x_1,\ldots,x_n)^\top$:
$$\mathbf{f} = (f(x_1),\ldots,f(x_n))^\top \sim \mathcal{N}(\mathbf{m}, \mathbf{K})$$
where $\mathbf{K}_{ij} = k(x_i, x_j)$.

**Common kernels:**

| Kernel | Formula | Properties |
|---|---|---|
| RBF / Squared Exponential | $\sigma^2 \exp\!\left(-\frac{(x-x')^2}{2\ell^2}\right)$ | Infinitely differentiable; smooth |
| Matérn 3/2 | $\sigma^2(1 + \frac{\sqrt 3 r}{\ell})e^{-\sqrt 3 r/\ell}$ | Once differentiable |
| Matérn 1/2 | $\sigma^2 e^{-r/\ell}$ | Continuous but not differentiable |
| Periodic | $\sigma^2 \exp(-\frac{2\sin^2(\pi r/p)}{\ell^2})$ | Periodic functions |
| Linear | $\sigma^2 (x^\top x')$ | Linear functions only |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial.distance import cdist

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

def rbf_kernel(X1, X2, length_scale=1.0, sigma_f=1.0):
    D = cdist(X1.reshape(-1,1), X2.reshape(-1,1))**2
    return sigma_f**2 * np.exp(-0.5 * D / length_scale**2)

def matern32_kernel(X1, X2, length_scale=1.0, sigma_f=1.0):
    D = cdist(X1.reshape(-1,1), X2.reshape(-1,1), metric='euclidean')
    return sigma_f**2 * (1 + np.sqrt(3)*D/length_scale) * np.exp(-np.sqrt(3)*D/length_scale)

def periodic_kernel(X1, X2, length_scale=1.0, sigma_f=1.0, period=2.0):
    D = cdist(X1.reshape(-1,1), X2.reshape(-1,1), metric='euclidean')
    return sigma_f**2 * np.exp(-2 * np.sin(np.pi*D/period)**2 / length_scale**2)

# ── GP prior samples: different kernels encode different smoothness ────────
x_test = np.linspace(-5, 5, 200)
kernels = [
    ('RBF (ℓ=1)',          rbf_kernel(x_test, x_test, 1.0)),
    ('RBF (ℓ=0.3)',        rbf_kernel(x_test, x_test, 0.3)),
    ('Matérn 3/2 (ℓ=1)',   matern32_kernel(x_test, x_test, 1.0)),
    ('Periodic (p=2)',      periodic_kernel(x_test, x_test, 1.0, period=2.0)),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, (name, K) in zip(axes.flat, kernels):
    K  = K + 1e-6 * np.eye(len(x_test))  # jitter for stability
    L  = np.linalg.cholesky(K)
    for _ in range(5):
        f = L @ np.random.standard_normal(len(x_test))
        ax.plot(x_test, f, lw=1.5, alpha=0.7)
    # Mean ± 2SD band
    sd = np.sqrt(np.diag(K))
    ax.fill_between(x_test, -2*sd, 2*sd, alpha=0.1, color='gray', label='±2 SD')
    ax.set_title(f'GP prior samples — {name}\nKernel encodes smoothness assumptions')
    ax.set_xlabel('x'); ax.set_ylabel('f(x)')

plt.suptitle('Gaussian Process Prior: Different Kernels = Different Function Families', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. GP regression (Kriging)

Given noisy observations $y_i = f(x_i) + \varepsilon_i$, $\varepsilon_i \sim \mathcal{N}(0, \sigma_n^2)$, the posterior process is:

$$f \mid \mathbf{x}, \mathbf{y} \sim \mathcal{GP}(m^*, k^*)$$

$$m^*(\mathbf{x}_*) = \mathbf{k}_*^\top (\mathbf{K} + \sigma_n^2\mathbf{I})^{-1}\mathbf{y}$$

$$k^*(\mathbf{x}_*, \mathbf{x}_*') = k(\mathbf{x}_*, \mathbf{x}_*') - \mathbf{k}_*^\top(\mathbf{K} + \sigma_n^2\mathbf{I})^{-1}\mathbf{k}_*'$$

where $\mathbf{k}_* = [k(x_*, x_1), \ldots, k(x_*, x_n)]^\top$ and $\mathbf{K}_{ij} = k(x_i, x_j)$.

**Complexity:** $O(n^3)$ for training, $O(n^2)$ for prediction — the main scalability challenge.

In [ ]:
# ── GP regression: posterior mean and uncertainty ─────────────────────────
def gp_posterior(X_train, y_train, X_test, kernel_fn, sigma_n=0.3):
    """Compute GP posterior mean and covariance."""
    K       = kernel_fn(X_train, X_train) + sigma_n**2 * np.eye(len(X_train))
    K_star  = kernel_fn(X_train, X_test)
    K_ss    = kernel_fn(X_test, X_test)
    L       = np.linalg.cholesky(K + 1e-6*np.eye(len(K)))
    alpha   = np.linalg.solve(L.T, np.linalg.solve(L, y_train))
    mu_star = K_star.T @ alpha
    v       = np.linalg.solve(L, K_star)
    cov_star = K_ss - v.T @ v
    return mu_star, cov_star

# True function
f_true = lambda x: np.sin(x) + 0.5*np.cos(2*x)

np.random.seed(42)
X_train = np.array([-4, -2.5, -1, 0.5, 2, 3.5])
y_train = f_true(X_train) + np.random.normal(0, 0.3, len(X_train))
X_test  = np.linspace(-5.5, 5.5, 300)

# Compare RBF kernels with different length scales
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, (ls, label) in zip(axes, [(0.5, 'ℓ=0.5 (too wiggly)'), (1.5, 'ℓ=1.5 (good fit)'), (5.0, 'ℓ=5 (too smooth)')]):
    k_fn = lambda X1, X2, l=ls: rbf_kernel(X1, X2, length_scale=l, sigma_f=1.0)
    mu, cov = gp_posterior(X_train, y_train, X_test, k_fn, sigma_n=0.3)
    sd = np.sqrt(np.maximum(np.diag(cov), 0))

    ax.plot(X_test, f_true(X_test), 'k--', lw=1.5, label='True f(x)')
    ax.plot(X_test, mu, color='#4361ee', lw=2.5, label='GP posterior mean')
    ax.fill_between(X_test, mu-2*sd, mu+2*sd, alpha=0.2, color='#4361ee', label='±2 SD')
    ax.scatter(X_train, y_train, color='#f72585', s=70, zorder=5, label='Training data')
    ax.set_title(f'RBF kernel, {label}\n$\\sigma_n=0.3$')
    ax.set_xlabel('x'); ax.set_ylabel('f(x)')
    ax.legend(fontsize=7)

plt.suptitle('GP Regression: Effect of Kernel Length Scale\n'
             'Length scale controls smoothness — must be learned from data',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Hyperparameter learning via marginal likelihood

The **log marginal likelihood** for GP regression:
$$\log p(\mathbf{y} \mid \mathbf{x}, \boldsymbol{\phi}) = -\frac{1}{2}\mathbf{y}^\top(\mathbf{K}_\phi + \sigma_n^2\mathbf{I})^{-1}\mathbf{y} - \frac{1}{2}\log|\mathbf{K}_\phi + \sigma_n^2\mathbf{I}| - \frac{n}{2}\log 2\pi$$

Maximizing over hyperparameters $\boldsymbol{\phi} = (\ell, \sigma_f, \sigma_n)$ balances:
- **Data fit** ($-\mathbf{y}^\top K^{-1}\mathbf{y}$): prefers complex models
- **Complexity penalty** ($\log|K|$): penalizes complex models

This automatic Occam's razor is a key advantage of Bayesian model selection.

## 4. Key takeaways

| Concept | Statement |
|---|---|
| GP = prior over functions | Any finite evaluation is multivariate Gaussian |
| Kernel = inductive bias | RBF: smooth; Matérn: rougher; Periodic: cyclical |
| GP posterior | Closed form: Gaussian, $O(n^3)$ |
| Hyperparameter learning | Maximize marginal likelihood — automatic complexity control |
| Uncertainty | GP gives credible bands for predictions |

**Next:** notebook 12 — Bayesian nonparametrics: Dirichlet processes, infinite mixture models.